# Ek analiz 2: katılımcı eşleştirmesi (parmak izi) ve normalizasyon duyarlılığı

**Çalışma zamanı > Tümünü çalıştır** (yaklaşık 20–40 dakika). Bitince **Dosya > Kaydet**.


In [ ]:
# 1) Gen setleri (MSigDB: SAUL_SEN_MAYO ve FRIDMAN_SENESCENCE_UP)
writeLines(c("ACVR1B", "ANG", "ANGPT1", "ANGPTL4", "AREG", "AXL", "BEX3", "BMP2", "BMP6", "C3", "CCL1", "CCL13", "CCL16", "CCL2", "CCL20", "CCL24", "CCL26", "CCL3", "CCL4", "CCL5", "CCL7", "CCL8", "CD55", "CD9", "CSF1", "CSF2", "CSF2RB", "CST4", "CTNNB1", "CTSB", "CXCL1", "CXCL10", "CXCL12", "CXCL16", "CXCL2", "CXCL3", "CXCL8", "CXCR2", "DKK1", "EDN1", "EGF", "EGFR", "EREG", "ESM1", "ETS2", "FAS", "FGF1", "FGF2", "FGF7", "GDF15", "GEM", "GMFG", "HGF", "HMGB1", "ICAM1", "ICAM3", "IGF1", "IGFBP1", "IGFBP2", "IGFBP3", "IGFBP4", "IGFBP5", "IGFBP6", "IGFBP7", "IL10", "IL13", "IL15", "IL18", "IL1A", "IL1B", "IL2", "IL32", "IL6", "IL6ST", "IL7", "INHA", "IQGAP2", "ITGA2", "ITPKA", "JUN", "KITLG", "LCP1", "MIF", "MMP1", "MMP10", "MMP12", "MMP13", "MMP14", "MMP2", "MMP3", "MMP9", "NAP1L4", "NRG1", "PAPPA", "PECAM1", "PGF", "PIGF", "PLAT", "PLAU", "PLAUR", "PTBP1", "PTGER2", "PTGES", "RPS6KA5", "SCAMP4", "SELPLG", "SEMA3F", "SERPINB4", "SERPINE1", "SERPINE2", "SPP1", "SPX", "TIMP2", "TNF", "TNFRSF10C", "TNFRSF11B", "TNFRSF1A", "TNFRSF1B", "TUBGCP2", "VEGFA", "VEGFC", "VGF", "WNT16", "WNT2", "CCL3L1"), "senmayo.txt")
writeLines(c("OPTN", "CDKN1A", "CDKN1C", "CDKN2A", "CDKN2B", "CDKN2D", "CITED2", "HTATIP2", "HBS1L", "RAB31", "ESM1", "HPS5", "FILIP1L", "CLTB", "COL1A2", "CRYAB", "CCN2", "CYP1B1", "F3", "ALDH1A3", "FN1", "TES", "GSN", "GUK1", "NRG1", "HSPA2", "IGSF3", "IFI16", "IFNG", "IGFBP1", "IGFBP2", "IGFBP3", "IGFBP4", "IGFBP5", "IGFBP6", "IGFBP7", "ING1", "IRF5", "IRF7", "RHOB", "MDM2", "MMP1", "NDN", "NME2", "SERPINE1", "SERPINB2", "MAP2K3", "RGL2", "RAB5B", "RAB13", "RABGGTA", "RAC1", "RBL2", "CCND1", "RRAS", "S100A11", "SMURF2", "SMPD1", "SOD1", "SPARC", "STAT1", "TFAP2A", "TGFB1I1", "THBS1", "TNFAIP2", "TNFAIP3", "TP53", "VIM", "MAP1LC3B", "AOPEP", "TSPYL5", "PEA15", "CREG1", "EIF2S2", "CXCL14", "CD44", "ISG15"), "fridman_up.txt")
cat("Gen seti dosyaları yazıldı.\n")
cat(R.version.string, "\n")

In [ ]:
# 2) Fonksiyonlar (analiz çalıştırılmaz)
###############################################################################
# PİLOT ANALİZ: Kilo kaybı sonrası yağ dokusu senesans imzası
# Veri setleri: GSE199063 (bariatrik cerrahi; öncesi, 2. yıl, 5. yıl)
#               GSE77962  (diyet; öncesi, kilo kaybı sonrası, idame sonrası)
#
# Soru: SenMayo senesans skoru kilo kaybından sonra düşüyor mu?
#       Düşüş, rastgele gen setlerinden beklenenden büyük mü?
#
# Kullanım: R veya RStudio'da bu dosyayı açın ve tamamını çalıştırın
#           (RStudio: Ctrl+Shift+Enter / Cmd+Shift+Enter).
# Tüm çıktılar "senesans_pilot_cikti" klasörüne kaydedilir.
###############################################################################

## ---------------------------------------------------------------------------
## 0) AYARLAR
## ---------------------------------------------------------------------------
CIKTI_KLASORU <- "senesans_pilot_cikti"
VERI_SETLERI  <- c("GSE199063", "GSE77962", "GSE59034", "GSE65540")
RNASEQ_SETLERI <- c("GSE65540")   # ifade verisi NCBI'nin hazırladığı sayım matrisinden okunur
RASTGELE_SET_SAYISI <- 1000   # özgüllük testi için rastgele gen seti sayısı
DUYARLILIK <- TRUE            # HPA eşik ve tek gen çıkarma duyarlılık analizleri
PROB_BIRLESTIRME <- "en_yuksek"  # aynı gene ait problar: "en_yuksek" (ortalaması en yüksek prob) veya "ortalama"
set.seed(2026)

# Otomatik algılama başarısız olursa buraya sütun adlarını elle yazın.
# Sütun adlarını "01_metadata_<GSE>.txt" dosyasında görebilirsiniz.
# Örnek: ELLE_AYAR$GSE199063 <- list(zaman = "time point:ch1", kisi = "subject:ch1",
#                                    baslangic = "before")
# Makrofaj işaret genleri (SenMayo ile ortak gen içermez)
MAKROFAJ_GENLERI <- c("CD68", "CD14", "CD163", "ITGAM", "ITGAX", "TREM2", "LYVE1", "MRC1", "MSR1",
                      "C1QA", "C1QB", "C1QC", "CSF1R", "FCGR3A", "LIPA", "PLA2G7", "ADGRE1", "MS4A7",
                      "TYROBP", "FCER1G", "LAPTM5", "AIF1", "CD86", "SIGLEC1")
T_HUCRE_GENLERI <- c("CD3D", "CD3E", "CD3G", "CD2", "CD5", "CD6", "CD8A", "CD8B", "CD247", "LCK", "ZAP70", "GZMK", "TRAT1", "CD96")
MAST_GENLERI    <- c("TPSAB1", "TPSB2", "CPA3", "MS4A2", "KIT", "HDC", "HPGDS", "CMA1", "SIGLEC6", "CLC")
ADIPOSIT_GENLERI <- c("ADIPOQ", "PLIN1", "PLIN4", "LEP", "CIDEA", "CIDEC", "LIPE", "PNPLA2", "FABP4", "GPD1", "TUSC5", "PPARG")
MAKROFAJ_KOR_ESIGI <- 0.5   # bir SenMayo geni herhangi bir bağışıklık skoruyla bu düzeyin üstünde ilişkiliyse "immün" sayılır   # bu korelasyonun üstündeki SenMayo genleri "makrofajla ilişkili" sayılır
# Klasik senesans belirteçleri (LMNB1 senesansta azalır)
TEMEL_SENESANS_GENLERI <- c("CDKN1A", "CDKN2A", "CDKN2B", "TP53", "LMNB1", "GLB1", "SERPINE1", "MKI67")

ELLE_AYAR <- list(
  GSE199063 = list(zaman = "time_point:ch1", kisi = "subject_id:ch1",
                   baslangic = "Baseline", kontrol = "Control"),
  GSE77962  = list(zaman = "time point:ch1", kisi = "subject_id:ch1",
                   baslangic = "at study start", grup = "treatment:ch1")
)

## ---------------------------------------------------------------------------
## 1) PAKETLER
## ---------------------------------------------------------------------------
# Gen seti dosyaları (senmayo.txt, fridman_up.txt) klasörde varsa msigdbr gerekmez
DOSYADAN_GEN_SETI <- file.exists("senmayo.txt")
cran <- c("ggplot2", if (!DOSYADAN_GEN_SETI) "msigdbr")
bioc <- c("GEOquery", "limma", "Biobase")
eksik_cran <- cran[!sapply(cran, requireNamespace, quietly = TRUE)]
if (length(eksik_cran)) install.packages(eksik_cran)
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
eksik_bioc <- bioc[!sapply(bioc, requireNamespace, quietly = TRUE)]
if (length(eksik_bioc)) BiocManager::install(eksik_bioc, update = FALSE, ask = FALSE)
# msigdbr 10+ sürümünde gen setleri ayrı "msigdbdf" paketinde
if (!DOSYADAN_GEN_SETI && packageVersion("msigdbr") >= "10.0.0" && !requireNamespace("msigdbdf", quietly = TRUE)) {
  install.packages("msigdbdf",
                   repos = c("https://igordot.r-universe.dev", "https://cloud.r-project.org"))
}

suppressPackageStartupMessages({
  library(GEOquery); library(limma); library(Biobase); library(ggplot2)
})
options(timeout = 3600)  # büyük dosyalar için indirme süresi
dir.create(CIKTI_KLASORU, showWarnings = FALSE)
yol <- function(...) file.path(CIKTI_KLASORU, paste0(...))

## ---------------------------------------------------------------------------
## 2) GEN SETLERİ (MSigDB)
## ---------------------------------------------------------------------------
gen_setlerini_al <- function() {
  oku <- function(f) if (file.exists(f)) unique(trimws(readLines(f))) else NULL
  if (DOSYADAN_GEN_SETI) {
    setler <- list(SenMayo = oku("senmayo.txt"), Fridman_SEN_UP = oku("fridman_up.txt"))
    return(setler[!sapply(setler, is.null)])
  }
  library(msigdbr)
  m <- tryCatch(msigdbr(species = "Homo sapiens", collection = "C2"),
                error = function(e) msigdbr(species = "Homo sapiens", category = "C2"))
  bul <- function(desen) {
    ad <- unique(m$gs_name[grepl(desen, m$gs_name)])
    if (!length(ad)) return(NULL)
    unique(m$gene_symbol[m$gs_name == ad[1]])
  }
  setler <- list(SenMayo = bul("SEN_?MAYO"), Fridman_SEN_UP = bul("^FRIDMAN_SENESCENCE_UP$"))
  setler <- setler[!sapply(setler, is.null)]
  if (is.null(setler$SenMayo))
    stop("SenMayo gen seti bulunamadı. Klasöre senmayo.txt dosyası koyun.")
  setler
}

## ---------------------------------------------------------------------------
## 3) VERİ İNDİRME VE GEN SEMBOLÜNE ÇEVİRME
## ---------------------------------------------------------------------------
sembol_bul <- function(fd) {
  aday <- c("Gene Symbol", "Gene symbol", "GENE_SYMBOL", "Symbol", "SYMBOL", "gene_symbol",
            "GeneSymbol")
  s <- intersect(aday, colnames(fd))
  if (length(s)) return(trimws(sub(" ///.*", "", as.character(fd[[s[1]]]))))
  if ("gene_assignment" %in% colnames(fd)) {         # Affymetrix Gene/Clariom dizileri
    return(trimws(vapply(strsplit(as.character(fd$gene_assignment), " // "),
                         function(x) if (length(x) >= 2) x[2] else NA_character_, "")))
  }
  NULL
}

# GEO web sayfası (www.ncbi.nlm.nih.gov) otomatik isteklere doğrulama ekranı gösterebildiği için
# dosyalar doğrudan NCBI dosya sunucusundan (ftp.ncbi.nlm.nih.gov) indirilir.
ftp_klasor <- function(id, tur) {             # GSE77962 -> .../series/GSE77nnn/GSE77962/
  n <- sub("^[A-Z]+", "", id)
  kok <- if (nchar(n) > 3) substr(n, 1, nchar(n) - 3) else ""
  sprintf("https://ftp.ncbi.nlm.nih.gov/geo/%s/%s%snnn/%s/", tur, substr(id, 1, 3), kok, id)
}
INDIRME_KLASORU <- "geo_indirilen"
dir.create(INDIRME_KLASORU, showWarnings = FALSE)
indir <- function(url) {
  hedef <- file.path(INDIRME_KLASORU, basename(url))
  if (file.exists(hedef) && file.size(hedef) > 0) return(hedef)
  for (u in c(url, sub("^https://", "ftp://", url))) {
    ok <- tryCatch({ download.file(u, hedef, mode = "wb", quiet = TRUE); file.size(hedef) > 0 },
                   error = function(e) { message("    indirilemedi: ", u, " (", conditionMessage(e), ")"); FALSE },
                   warning = function(w) { message("    indirilemedi: ", u, " (", conditionMessage(w), ")"); FALSE })
    if (isTRUE(ok)) return(hedef)
  }
  if (file.exists(hedef)) file.remove(hedef)
  NULL
}
dizin_listesi <- function(url, desen) {
  html <- tryCatch(paste(readLines(url, warn = FALSE), collapse = ""), error = function(e) "")
  unique(regmatches(html, gregexpr(desen, html))[[1]])
}
# Platform tablosundan yalnızca prob ID ve gen sembolünü, dosyayı belleğe almadan (zcat + awk) çıkarır.
# Clariom D gibi büyük platformların tam tablosu birkaç GB yer kaplayabildiği için bu şekilde okunur.
platform_sembol_oku <- function(dosya) {
  baslik <- system(sprintf("zcat '%s' | grep -m1 -A1 '^!platform_table_begin' | tail -n 1", dosya), intern = TRUE)
  if (!length(baslik)) return(NULL)
  sut <- strsplit(baslik, "\t")[[1]]
  aday <- c("Gene Symbol", "Gene symbol", "GENE_SYMBOL", "Symbol", "SYMBOL", "gene_symbol", "GeneSymbol")
  if (any(aday %in% sut)) { i <- match(aday[aday %in% sut][1], sut); tur <- "sembol"
  } else if ("gene_assignment" %in% sut) { i <- match("gene_assignment", sut); tur <- "ga"
  } else return(NULL)
  awk <- if (tur == "ga")
    sprintf("f&&/^!platform_table_end/{exit} f{n=split($%d,a,\" // \"); print $1 \"\\t\" (n>=2?a[2]:\"\")} /^!platform_table_begin/{f=1; getline}", i)
  else
    sprintf("f&&/^!platform_table_end/{exit} f{s=$%d; sub(/ \\/\\/\\/.*/,\"\",s); print $1 \"\\t\" s} /^!platform_table_begin/{f=1; getline}", i)
  komut <- sprintf("zcat '%s' | awk -F'\\t' '%s'", dosya, awk)
  tab <- read.delim(pipe(komut), header = FALSE, quote = "", comment.char = "",
                    col.names = c("ID", "sembol"), colClasses = "character")
  tab$sembol <- trimws(tab$sembol)
  tab
}

# GEOquery'nin yeni sürümleri (Bioconductor 3.24+) SummarizedExperiment döndürür; eski sürümler
# ExpressionSet döndürür. İki durumda da ExpressionSet'e çevrilir.
geo_oku <- function(f) {
  es <- tryCatch(getGEO(filename = f, getGPL = FALSE, returnType = "ExpressionSet"),
                 error = function(e) getGEO(filename = f, getGPL = FALSE))
  if (inherits(es, "SummarizedExperiment")) {
    m <- as.matrix(SummarizedExperiment::assay(es))
    pd <- as.data.frame(SummarizedExperiment::colData(es), optional = TRUE)
    rownames(pd) <- colnames(m)
    es <- ExpressionSet(assayData = m, phenoData = AnnotatedDataFrame(pd))
  }
  pd <- pData(es)
  if (!any(grepl(":ch1$", colnames(pd)))) {        # "anahtar: değer" sütunlarından :ch1 sütunları üret
    for (c in grep("^characteristics", colnames(pd), value = TRUE)) {
      v <- as.character(pd[[c]]); k <- unique(sub(":.*$", "", v[grepl(":", v)]))
      if (length(k) == 1) pd[[paste0(k, ":ch1")]] <- trimws(sub("^[^:]*:", "", v))
    }
    pData(es) <- pd
  }
  es
}

gse_yukle <- function(acc) {
  message("\n>>> ", acc, " indiriliyor...")
  url <- paste0(ftp_klasor(acc, "series"), "matrix/")
  yerel <- Filter(Negate(is.null), list(indir(paste0(url, acc, "_series_matrix.txt.gz"))))
  if (!length(yerel)) {   # birden çok platformlu seriler: dosya adları -GPLxxx içerir
    dosyalar <- suppressWarnings(dizin_listesi(url, paste0(acc, "[^\"<>/]*_series_matrix\\.txt\\.gz")))
    yerel <- Filter(Negate(is.null), lapply(paste0(url, dosyalar), indir))
  }
  if (!length(yerel)) stop(acc, ": seri matrisi indirilemedi (bağlantı tanı mesajlarına bakın).")
  liste <- lapply(yerel, geo_oku)
  en_buyuk <- which.max(sapply(liste, ncol))
  es <- liste[[en_buyuk]]
  if (nrow(exprs(es)) == 0)
    stop(acc, ": seri matrisinde ifade verisi yok (muhtemelen RNA-seq). Bu pilot yalnızca mikroarray içindir.")

  bas <- readLines(gzfile(yerel[[en_buyuk]]), n = 200, warn = FALSE)
  gpl_id <- gsub("[^A-Z0-9]", "", sub("^!Series_platform_id\t", "", grep("^!Series_platform_id", bas, value = TRUE)[1]))
  message("    Platform: ", gpl_id, " | ", nrow(es), " prob x ", ncol(es), " örnek")
  gpl <- NULL
  for (u in c(paste0(ftp_klasor(gpl_id, "platforms"), "annot/", gpl_id, ".annot.gz"),
              paste0(ftp_klasor(gpl_id, "platforms"), "soft/", gpl_id, "_family.soft.gz"))) {
    f <- indir(u)
    if (!is.null(f)) gpl <- platform_sembol_oku(f)
    if (!is.null(gpl) && mean(gpl$sembol != "" & gpl$sembol != "---") > 0.1) break
  }
  if (is.null(gpl)) stop(acc, ": platform anotasyonu (", gpl_id, ") indirilemedi veya gen sembolü sütunu yok.")
  s <- gpl$sembol[match(rownames(es), gpl$ID)]
  rm(gpl); invisible(gc())
  message("    Gen sembolü eşlenen prob oranı: ", round(100 * mean(!is.na(s) & s != "" & s != "---"), 1), "%")

  x <- exprs(es)
  if (max(x, na.rm = TRUE) > 100) x <- log2(pmax(x, 0) + 1)   # log2 değilse dönüştür
  ok <- !is.na(s) & s != "" & s != "---"
  x <- x[ok, , drop = FALSE]; s <- s[ok]
  # Aynı gene ait birden çok prob: ortalaması en yüksek prob seçilir
  if (identical(PROB_BIRLESTIRME, "ortalama")) {
    x <- rowsum(x, s, na.rm = TRUE) / as.vector(table(s)[sort(unique(s))])
  } else {
    ort <- rowMeans(x, na.rm = TRUE)
    sira <- order(s, -ort)
    x <- x[sira, , drop = FALSE]; s <- s[sira]
    x <- x[!duplicated(s), , drop = FALSE]; rownames(x) <- s[!duplicated(s)]
  }
  message("    Analize giren gen sayısı: ", nrow(x), " (prob birleştirme: ", PROB_BIRLESTIRME, ")")
  list(acc = acc, x = x, pd = pData(es))
}

# RNA-seq verisi: ifade NCBI'nin GEO için hazırladığı ham sayım matrisinden, örnek bilgileri seri matrisinden okunur.
rnaseq_yukle <- function(acc) {
  message("\n>>> ", acc, " (RNA-seq) indiriliyor...")
  f <- indir(paste0(ftp_klasor(acc, "series"), "matrix/", acc, "_series_matrix.txt.gz"))
  if (is.null(f)) stop(acc, ": seri matrisi (örnek bilgileri) indirilemedi.")
  pd <- pData(geo_oku(f))
  kok <- "https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts"
  say_ad <- paste0(acc, "_raw_counts_GRCh38.p13_NCBI.tsv.gz"); ann_ad <- "Human.GRCh38.p13.annot.tsv.gz"
  say_f <- file.path(INDIRME_KLASORU, say_ad); ann_f <- file.path(INDIRME_KLASORU, ann_ad)
  gecerli <- function(dosya, ilk) file.exists(dosya) &&
    isTRUE(tryCatch(startsWith(readLines(gzfile(dosya), n = 1), ilk), error = function(e) FALSE))
  indir_dene <- function(url, hedef) tryCatch(download.file(url, hedef, mode = "wb", quiet = TRUE),
    error = function(e) message("    indirilemedi: ", url, " (", conditionMessage(e), ")"),
    warning = function(w) message("    indirilemedi: ", url, " (", conditionMessage(w), ")"))
  if (!gecerli(say_f, "GeneID")) indir_dene(paste0(kok, "&acc=", acc, "&format=file&file=", say_ad), say_f)
  if (!gecerli(say_f, "GeneID"))
    stop(acc, ": NCBI sayım dosyası alınamadı (", say_ad, "). Sunucu erişimi engellemiş olabilir.")
  for (u in c(paste0(kok, "&acc=", acc, "&format=file&file=", ann_ad), paste0(kok, "&format=file&file=", ann_ad)))
    if (!gecerli(ann_f, "GeneID")) indir_dene(u, ann_f)
  say <- read.delim(gzfile(say_f), check.names = FALSE, row.names = 1)
  if (gecerli(ann_f, "GeneID")) {
    ann <- read.delim(gzfile(ann_f), quote = "", check.names = FALSE, comment.char = "")
    sembol <- as.character(ann$Symbol[match(rownames(say), ann$GeneID)])
  } else {                                     # yedek: Bioconductor org.Hs.eg.db ile Entrez -> sembol
    message("    NCBI anotasyonu alınamadı; org.Hs.eg.db ile eşleniyor (ilk seferde kurulum birkaç dakika sürer)...")
    if (!requireNamespace("org.Hs.eg.db", quietly = TRUE)) BiocManager::install("org.Hs.eg.db", update = FALSE, ask = FALSE)
    sembol <- unname(suppressMessages(AnnotationDbi::mapIds(org.Hs.eg.db::org.Hs.eg.db, keys = rownames(say),
                                                        column = "SYMBOL", keytype = "ENTREZID", multiVals = "first")))
  }
  say <- as.matrix(say); ortak <- intersect(colnames(say), rownames(pd))
  if (length(ortak) < 4) stop(acc, ": sayım matrisi ile örnek bilgileri eşleşmedi.")
  say <- say[, ortak]; pd <- pd[ortak, , drop = FALSE]
  kutup <- colSums(say)
  if (exists("RNASEQ_TMM") && isTRUE(RNASEQ_TMM)) {   # duyarlılık: edgeR TMM normalizasyonu
    if (!requireNamespace("edgeR", quietly = TRUE)) BiocManager::install("edgeR", update = FALSE, ask = FALSE)
    kutup <- kutup * edgeR::calcNormFactors(edgeR::DGEList(say))$samples$norm.factors
    message("    TMM normalizasyonu uygulandı")
  }
  cpm <- t(t(say) / kutup * 1e6)
  tut <- rowMeans(cpm > 1) >= 0.2 & !is.na(sembol) & sembol != ""
  x <- log2(cpm[tut, , drop = FALSE] + 1); s <- sembol[tut]
  ort <- rowMeans(x); sira <- order(s, -ort); x <- x[sira, , drop = FALSE]; s <- s[sira]
  x <- x[!duplicated(s), , drop = FALSE]; rownames(x) <- s[!duplicated(s)]
  message("    ", ncol(x), " örnek | analize giren gen sayısı: ", nrow(x), " (log2 CPM)")
  list(acc = acc, x = x, pd = pd, rnaseq = TRUE)
}

## ---------------------------------------------------------------------------
## 4) METADATA: ZAMAN VE KİŞİ SÜTUNLARINI BULMA
## ---------------------------------------------------------------------------
metadata_ozeti_yaz <- function(acc, pd) {
  sut <- grep(":ch1$|^characteristics|^title$|^source_name", colnames(pd), value = TRUE)
  dosya <- yol("01_metadata_", acc, ".txt")
  sink(dosya)
  cat("Veri seti:", acc, " | örnek sayısı:", nrow(pd), "\n\n")
  for (c in sut) {
    u <- unique(as.character(pd[[c]]))
    cat("SÜTUN:", c, " (", length(u), " farklı değer)\n  ", paste(head(u, 8), collapse = " | "),
        if (length(u) > 8) " | ..." else "", "\n\n")
  }
  sink()
  dosya
}

sutunlari_algila <- function(acc, pd) {
  if (!is.null(ELLE_AYAR[[acc]])) return(ELLE_AYAR[[acc]])
  sut <- grep(":ch1$", colnames(pd), value = TRUE)
  zaman_deseni <- "before|after|baseline|pre|post|year|month|week|visit|time|t0|t1|t2|t3|follow|maint|cid|surgery|control|lean|never|non.?obese"
  zaman <- NULL; kisi <- NULL
  for (c in sut) {
    v <- tolower(as.character(pd[[c]])); n <- length(unique(v))
    if (is.null(zaman) && n >= 2 && n <= 5 && mean(grepl(zaman_deseni, v)) > 0.6) zaman <- c
  }
  for (c in sut) {
    n <- length(unique(pd[[c]]))
    if (is.null(zaman) && n >= 2 && n <= 5 && grepl("time|visit|point|stage|cid|state|status|group|condition", tolower(c))) zaman <- c
  }
  for (c in sut) {
    v <- as.character(pd[[c]]); n <- length(unique(v))
    if (is.null(kisi) && grepl("subject|patient|individual|donor|participant|^id|code|pair", tolower(c)) &&
        n < length(v) && n > 2)
      kisi <- c
  }
  if (is.null(kisi)) {   # başlıktan kişi kimliği çıkarmayı dene (ör. "Subject 12, before")
    t <- as.character(pd$title)
    k <- sub(".*?(subject|patient|individual|id|p)[ _#-]*([0-9]+).*", "\\2", t, ignore.case = TRUE, perl = TRUE)
    if (mean(k != t) > 0.8 && length(unique(k)) < length(k)) { pd$..kisi <- k; kisi <- "..kisi" }
  }
  if (is.null(zaman)) return(NULL)
  if (is.null(kisi) && !is.null(pd$title)) {   # başlıktan zaman ifadelerini silip kalanı kişi kimliği say
    t <- tolower(as.character(pd$title))
    for (z in unique(tolower(as.character(pd[[zaman]])))) t <- gsub(z, " ", t, fixed = TRUE)
    t <- gsub("\\b(t[0-9]+|m[0-9]+|y[0-9]+|v[0-9]+|before|after|pre|post|baseline|follow.?up|surgery|op|month[s]?|year[s]?|[0-9]+ ?(m|mo|y|yr))\\b", " ", t, perl = TRUE)
    t <- gsub("[^a-z0-9]+", "_", t); t <- gsub("^_+|_+$", "", t)
    tab <- table(t)
    if (length(tab) < length(t) && length(tab) >= 3 && mean(tab >= 2) >= 0.5) { pd$..kisi <- t; kisi <- "..kisi" }
  }
  eslesmesiz <- is.null(kisi)
  if (eslesmesiz) { pd$..kisi <- rownames(pd); kisi <- "..kisi" }   # eşleştirme yok: her örnek ayrı kişi
  v <- as.character(pd[[zaman]])
  kon <- unique(v[grepl("control|lean|never|non.?obese", v, ignore.case = TRUE)])
  aday <- setdiff(unique(v), kon)
  bas <- aday[grepl("before|baseline|pre|start|^t0|t0$|cid ?1|week 0|^0 ?(month|year)", aday, ignore.case = TRUE)]
  list(zaman = zaman, kisi = kisi, baslangic = if (length(bas)) bas[1] else NULL,
       kontrol = if (length(kon) == 1) kon else NULL, eslesmesiz = eslesmesiz, pd = pd)
}

## ---------------------------------------------------------------------------
## 5) SKORLAR
## ---------------------------------------------------------------------------
# ssGSEA (Barbie ve ark. 2009; GSVA paketindeki uygulamayla birebir aynı sonuç verir)
ssgsea_skor <- function(x, setler, alpha = 0.25) {
  x <- x[rowSums(is.na(x)) == 0, , drop = FALSE]
  setler <- lapply(setler, function(g) intersect(g, rownames(x)))
  setler <- setler[sapply(setler, length) >= 5]
  sk <- sapply(seq_len(ncol(x)), function(j) {
    r <- rank(x[, j], ties.method = "average")
    o <- order(r, decreasing = TRUE); ro <- r[o]; gen_o <- rownames(x)[o]
    sapply(setler, function(g) {
      hit <- gen_o %in% g
      w <- ro^alpha * hit
      sum(cumsum(w) / sum(w) - cumsum(!hit) / sum(!hit))
    })
  })
  sk <- matrix(sk, nrow = length(setler), dimnames = list(names(setler), colnames(x)))
  sk / diff(range(sk))
}
z_skor <- function(x, genler) {          # basit, yorumlanması kolay ikinci skor
  g <- intersect(genler, rownames(x))
  z <- t(scale(t(x[g, , drop = FALSE])))
  colMeans(z, na.rm = TRUE)
}

## ---------------------------------------------------------------------------
## 5b) SenMayo GENLERİNİN SABİT HÜCRE TİPİ SINIFLANDIRMASI (Human Protein Atlas)
## ---------------------------------------------------------------------------
# Her SenMayo geni, HPA tek hücre RNA verisinde bağışıklık hücrelerinde mi yoksa yağ dokusunun
# stromal/adiposit hücrelerinde mi daha çok ifade edildiğine göre BİR KEZ sınıflandırılır ve aynı
# liste tüm kohortlarda kullanılır.
HPA_URLLERI <- c("https://v23.proteinatlas.org/download/rna_single_cell_type.tsv.zip",
                 "https://v24.proteinatlas.org/download/rna_single_cell_type.tsv.zip",
                 "https://www.proteinatlas.org/download/tsv/rna_single_cell_type.tsv.zip",
                 "https://www.proteinatlas.org/download/rna_single_cell_type.tsv.zip")
IMMUN_DESEN   <- "macrophage|monocyte|t-cell|t cell|b-cell|b cell|nk-cell|nk cell|mast|dendritic|granulocyte|neutrophil|plasma|kupffer|langerhans|hofbauer|microglial"
STROMAL_DESEN <- "adipocyte|fibroblast|endothelial|smooth muscle|mesothelial|pericyte"

hpa_siniflandir <- function(genler) {
  z <- file.path(INDIRME_KLASORU, "hpa_rna_single_cell_type.tsv.zip")
  if (!file.exists(z) || file.size(z) < 2000) {
    for (u in HPA_URLLERI) {
      ok <- tryCatch({ download.file(u, z, mode = "wb", quiet = TRUE); file.size(z) > 2000 },
                     error = function(e) FALSE, warning = function(w) FALSE)
      if (isTRUE(ok)) { message("    HPA tek hücre verisi indirildi: ", u); break }
      message("    HPA indirilemedi: ", u)
    }
  }
  if (!file.exists(z) || file.size(z) < 2000) return(NULL)
  icerik <- unzip(z, list = TRUE)$Name[1]
  h <- read.delim(unz(z, icerik), check.names = FALSE, stringsAsFactors = FALSE)
  cn <- tolower(colnames(h))
  gad <- colnames(h)[match("gene name", cn)]
  if (is.na(gad)) { message("    HPA dosyasında 'Gene name' sütunu yok; sınıflandırma atlandı."); return(NULL) }
  h <- h[h[[gad]] %in% genler, , drop = FALSE]
  if ("cell type" %in% cn) {                                   # uzun format
    ct <- colnames(h)[match("cell type", cn)]; nt <- colnames(h)[match("ntpm", cn)]
    m <- tapply(as.numeric(h[[nt]]), list(h[[gad]], h[[ct]]), max)
  } else {                                                     # geniş format
    say <- sapply(h, is.numeric); m <- as.matrix(h[, say, drop = FALSE]); rownames(m) <- h[[gad]]
    m <- m[!duplicated(rownames(m)), , drop = FALSE]
  }
  m[is.na(m)] <- 0
  ct <- tolower(colnames(m)); imm <- grepl(IMMUN_DESEN, ct); str <- grepl(STROMAL_DESEN, ct)
  if (sum(imm) < 3 || sum(str) < 2) { message("    HPA hücre tipleri tanınamadı; sınıflandırma atlandı."); return(NULL) }
  imm_max <- apply(m[, imm, drop = FALSE], 1, max); str_max <- apply(m[, str, drop = FALSE], 1, max)
  pay <- imm_max / (imm_max + str_max + 1e-9)
  data.frame(gen = rownames(m), immun_max_nTPM = round(imm_max, 1), stromal_max_nTPM = round(str_max, 1),
             immun_payi = round(pay, 3),
             sinif = ifelse(imm_max + str_max < 1, "dusuk_ifade",
                     ifelse(pay >= 0.6, "immun", ifelse(pay <= 0.4, "stromal_adiposit", "karisik"))),
             stringsAsFactors = FALSE)
}
SABIT_SINIF <- NULL   # çalıştırma bölümünde doldurulur

## ---------------------------------------------------------------------------
## 6) ANALİZ (bir veri seti için)
## ---------------------------------------------------------------------------
analiz_et <- function(d, setler) {
  acc <- d$acc
  metadata_ozeti_yaz(acc, d$pd)
  a <- sutunlari_algila(acc, d$pd)
  if (is.null(a) || is.null(a$baslangic)) {
    message("!!! ", acc, ": zaman/kişi sütunları otomatik bulunamadı. ",
            yol("01_metadata_", acc, ".txt"), " dosyasına bakıp ELLE_AYAR'ı doldurun.")
    return(NULL)
  }
  pd <- if (!is.null(a$pd)) a$pd else d$pd
  sutun_bul <- function(ad) {                      # sütun adı sürümler arasında küçük farklılık gösterebilir
    if (is.null(ad) || ad %in% colnames(pd)) return(ad)
    aday <- c(sub(":ch1$", "", ad), make.names(ad), make.names(sub(":ch1$", "", ad)))
    b <- aday[aday %in% colnames(pd)]
    if (length(b)) return(b[1])
    stop("'", ad, "' sütunu bulunamadı. Mevcut sütunlar: ", paste(colnames(pd), collapse = ", "))
  }
  a$zaman <- sutun_bul(a$zaman); a$kisi <- sutun_bul(a$kisi); a$grup <- sutun_bul(a$grup)
  zaman <- factor(as.character(pd[[a$zaman]]))
  dz <- setdiff(levels(zaman), c(a$baslangic, a$kontrol))            # sıra: başlangıç, takip(ler), kontrol
  zaman <- factor(zaman, levels = c(a$baslangic, dz, a$kontrol))
  kisi  <- factor(as.character(pd[[a$kisi]]))
  grup  <- if (!is.null(a$grup)) as.character(pd[[a$grup]]) else rep(NA_character_, nrow(pd))
  message("    Zaman sütunu: ", a$zaman, " (", paste(levels(zaman), collapse = ", "), ")")
  message("    Kişi sütunu : ", a$kisi, " (", nlevels(kisi), " kişi)")

  # --- Makrofaj skoru ve SenMayo'nun bağışıklık/bağışıklık dışı bölünmesi ---
  # Makrofaj seti SenMayo ile hiç ortak gen içermez (döngüsellik olmasın diye).
  setler$Makrofaj <- setdiff(MAKROFAJ_GENLERI, setler$SenMayo)
  setler$T_hucresi <- setdiff(T_HUCRE_GENLERI, setler$SenMayo)
  setler$Mast_hucresi <- setdiff(MAST_GENLERI, setler$SenMayo)
  setler$Adiposit <- setdiff(ADIPOSIT_GENLERI, setler$SenMayo)
  g_sen_tum <- intersect(setler$SenMayo, rownames(d$x))
  if (!is.null(SABIT_SINIF)) {                    # HPA'ya dayalı sabit sınıflandırma (tüm kohortlarda aynı)
    setler$SenMayo_immun      <- intersect(SABIT_SINIF$gen[SABIT_SINIF$sinif == "immun"], g_sen_tum)
    setler$SenMayo_immun_disi <- intersect(SABIT_SINIF$gen[SABIT_SINIF$sinif == "stromal_adiposit"], g_sen_tum)
    kaynak <- "HPA tek hücre verisi"
  } else {                                        # yedek: ilk kohortta korelasyonla belirlenip sabitlenir
    imm <- ssgsea_skor(d$x, setler[c("Makrofaj", "T_hucresi", "Mast_hucresi")])
    r_imm <- sapply(rownames(imm), function(k)
      apply(d$x[g_sen_tum, , drop = FALSE], 1, function(v) suppressWarnings(cor(v, imm[k, ]))))
    r_imm <- matrix(r_imm, nrow = length(g_sen_tum), dimnames = list(g_sen_tum, rownames(imm)))
    r_max <- apply(r_imm, 1, function(r) if (all(is.na(r))) NA else max(r, na.rm = TRUE))
    immun_mu <- !is.na(r_max) & r_max >= MAKROFAJ_KOR_ESIGI
    SABIT_SINIF <<- data.frame(gen = g_sen_tum, sinif = ifelse(immun_mu, "immun", "stromal_adiposit"))
    write.csv(SABIT_SINIF, yol("06_SenMayo_siniflandirma.csv"), row.names = FALSE)
    setler$SenMayo_immun <- g_sen_tum[immun_mu]; setler$SenMayo_immun_disi <- g_sen_tum[!immun_mu]
    kaynak <- paste0(acc, " korelasyonu (yedek yöntem; sonraki kohortlarda aynı liste kullanılır)")
  }
  message("    SenMayo sınıflandırması (", kaynak, "): ", length(setler$SenMayo_immun), " bağışıklık, ",
          length(setler$SenMayo_immun_disi), " stromal/adiposit geni")

  kapsam <- sapply(setler, function(g) sum(g %in% rownames(d$x)))
  sk <- ssgsea_skor(d$x, setler)
  df <- data.frame(ornek = colnames(d$x), kisi = kisi, zaman = zaman, grup = grup,
                   t(sk), SenMayo_z = z_skor(d$x, setler$SenMayo), check.names = FALSE)
  # Makrofaj içeriğinden arındırılmış SenMayo skoru (doğrusal regresyon artıkları)
  df$SenMayo_makrofaj_duzeltilmis <- unname(residuals(lm(df$SenMayo ~ df$Makrofaj)))
  # Klinik değişken (BMI) varsa ekle
  bmi_sut <- grep("(^|[^a-z])bmi([^a-z]|$)|body mass", colnames(pd), ignore.case = TRUE, value = TRUE)
  bmi_sut <- bmi_sut[!grepl("^characteristics", bmi_sut)]
  if (length(bmi_sut)) {
    deger <- as.character(pd[[bmi_sut[1]]])
    df$BMI <- suppressWarnings(as.numeric(sub(",", ".", gsub("[^0-9.,]", "", deger))))
    message("    BMI sütunu: ", bmi_sut[1], " (", sum(!is.na(df$BMI)), " dolu değer)")
  } else message("    BMI sütunu bulunamadı.")
  write.csv(df, yol("02_skorlar_", acc, ".csv"), row.names = FALSE)

  # --- Eşleştirilmiş karşılaştırmalar: her zaman noktası vs başlangıç ---
  skor_sut <- intersect(c(names(setler), "SenMayo_z", "SenMayo_makrofaj_duzeltilmis"), colnames(df))
  eslestir <- function(dd, grup_etiketi) {
    out <- list()
    for (zn in dz) {
      ortak <- intersect(dd$kisi[dd$zaman == a$baslangic], dd$kisi[dd$zaman == zn])
      for (s in skor_sut) {
        if (length(ortak) >= 3) {                       # eşleştirilmiş test
          once  <- dd[dd$zaman == a$baslangic, ]; sonra <- dd[dd$zaman == zn, ]
          once  <- once[match(ortak, once$kisi), s]; sonra <- sonra[match(ortak, sonra$kisi), s]
          fark <- sonra - once
          out[[length(out) + 1]] <- data.frame(
            veri_seti = acc, grup = grup_etiketi, karsilastirma = paste(zn, "vs", a$baslangic), skor = s,
            test = "eslestirilmis", n_cift = length(ortak), ort_degisim = mean(fark), cohen_dz = mean(fark) / sd(fark),
            p_eslestirilmis_t = t.test(sonra, once, paired = TRUE)$p.value,
            p_wilcoxon = suppressWarnings(wilcox.test(sonra, once, paired = TRUE)$p.value))
        } else {                                        # eşleştirme yoksa: bağımsız gruplar (Welch)
          x0 <- dd[dd$zaman == a$baslangic, s]; x1 <- dd[dd$zaman == zn, s]
          if (length(x0) < 3 || length(x1) < 3) next
          out[[length(out) + 1]] <- data.frame(
            veri_seti = acc, grup = grup_etiketi, karsilastirma = paste(zn, "vs", a$baslangic), skor = s,
            test = "eslestirilmemis", n_cift = min(length(x0), length(x1)), ort_degisim = mean(x1) - mean(x0),
            cohen_dz = (mean(x1) - mean(x0)) / sqrt((var(x1) + var(x0)) / 2),
            p_eslestirilmis_t = t.test(x1, x0)$p.value,
            p_wilcoxon = suppressWarnings(wilcox.test(x1, x0)$p.value))
        }
      }
    }
    do.call(rbind, out)
  }
  sonuc <- eslestir(df, "tum_orneklem")
  if (!is.null(a$grup))                       # alt grup analizi (ör. diyet tipi)
    for (g in unique(na.omit(df$grup))) sonuc <- rbind(sonuc, eslestir(df[df$grup %in% g, ], g))

  # --- Hiç obez olmamış kontrollerle karşılaştırma (eşleştirilmemiş, Welch t) ---
  kontrol_sonuc <- NULL
  if (!is.null(a$kontrol) && sum(df$zaman == a$kontrol) >= 3) {
    for (zn in c(a$baslangic, dz)) for (s in skor_sut) {
      x1 <- df[df$zaman == zn, s]; x0 <- df[df$zaman == a$kontrol, s]
      kontrol_sonuc <- rbind(kontrol_sonuc, data.frame(
        veri_seti = acc, karsilastirma = paste(zn, "vs", a$kontrol), skor = s,
        n_grup = length(x1), n_kontrol = length(x0), ort_fark = mean(x1) - mean(x0),
        cohen_d = (mean(x1) - mean(x0)) / sqrt((var(x1) + var(x0)) / 2),
        p_welch = t.test(x1, x0)$p.value))
    }
  }

  # --- Özgüllük: aynı büyüklükte rastgele gen setleri ---
  g_sen <- intersect(setler$SenMayo, rownames(d$x))
  ozgulluk <- list()
  for (zn in dz) {
    ortak <- intersect(df$kisi[df$zaman == a$baslangic], df$kisi[df$zaman == zn])
    if (length(ortak) < 3) next
    i0 <- match(ortak, df$kisi[df$zaman == a$baslangic]); i1 <- match(ortak, df$kisi[df$zaman == zn])
    s0 <- which(df$zaman == a$baslangic)[i0];             s1 <- which(df$zaman == zn)[i1]
    degisim <- function(genler) { z <- z_skor(d$x, genler); mean(z[s1] - z[s0]) }
    gercek <- degisim(g_sen)
    rast <- replicate(RASTGELE_SET_SAYISI, degisim(sample(rownames(d$x), length(g_sen))))
    ozgulluk[[zn]] <- data.frame(veri_seti = acc, karsilastirma = paste(zn, "vs", a$baslangic),
      gercek_degisim = gercek, rastgele_ortalama = mean(rast),
      ampirik_p_iki_yonlu = (sum(abs(rast - mean(rast)) >= abs(gercek - mean(rast))) + 1) / (RASTGELE_SET_SAYISI + 1))
  }
  ozgulluk <- do.call(rbind, ozgulluk)

  # --- Gen düzeyi: SenMayo genleri (limma, kişi bloklu eşleştirilmiş tasarım) ---
  tutulan <- kisi %in% names(which(table(kisi) >= 2))
  if (sum(tutulan) >= 6) {
    K <- droplevels(kisi[tutulan]); Z <- droplevels(zaman[tutulan])
    tasarim <- model.matrix(~ K + Z)
  } else {                                   # eşleştirme yok: kontrol grubu hariç basit grup karşılaştırması
    tutulan <- !is.na(zaman) & !(zaman %in% a$kontrol)
    Z <- droplevels(zaman[tutulan]); tasarim <- model.matrix(~ Z)
  }
  fit <- eBayes(lmFit(d$x[, tutulan], tasarim), trend = isTRUE(d$rnaseq))
  zaman_kat <- grep("^Z", colnames(tasarim))
  genler <- do.call(rbind, lapply(zaman_kat, function(k) {
    tt <- topTable(fit, coef = k, number = Inf)
    tt$gen <- rownames(tt)
    tt$karsilastirma <- paste(sub("^Z", "", colnames(tasarim)[k]), "vs", a$baslangic)
    tt$SenMayo_geni <- tt$gen %in% g_sen
    tt
  }))
  write.csv(genler[genler$SenMayo_geni, ], yol("04_SenMayo_genleri_", acc, ".csv"), row.names = FALSE)
  temel <- genler[genler$gen %in% TEMEL_SENESANS_GENLERI, c("gen", "karsilastirma", "logFC", "P.Value", "adj.P.Val")]
  write.csv(temel[order(temel$karsilastirma, temel$gen), ], yol("04_temel_senesans_genleri_", acc, ".csv"), row.names = FALSE)

  # --- Grafik ---
  p <- ggplot(df, aes(zaman, SenMayo, group = kisi)) +
    geom_line(alpha = 0.25, colour = "grey40") +
    geom_boxplot(aes(group = zaman), width = 0.35, fill = "#7FA7C9", alpha = 0.6, outlier.shape = NA) +
    geom_point(size = 1, alpha = 0.6) +
    labs(title = paste0(acc, ": SenMayo ssGSEA skoru"), x = NULL, y = "SenMayo skoru") +
    theme_bw(base_size = 12) + theme(axis.text.x = element_text(angle = 20, hjust = 1))
  if (!is.null(a$grup)) p <- p + facet_wrap(~ grup)
  ggsave(yol("05_grafik_", acc, ".png"), p, width = if (is.null(a$grup)) 6.5 else 9, height = 4.8, dpi = 300)

  # --- Doku bileşimine göre düzeltilmiş değişim ---
  # Eşleştirilmiş farklar üzerinde: Δskor = b0 + b1·ΔAdiposit + b2·ΔMakrofaj. b0, doku bileşimi
  # (adiposit ve makrofaj payı) değişmeseydi beklenecek skor değişimidir.
  bilesim <- NULL
  for (zn in dz) {
    ortak <- intersect(df$kisi[df$zaman == a$baslangic], df$kisi[df$zaman == zn])
    if (length(ortak) < 10 || !all(c("Adiposit", "Makrofaj") %in% colnames(df))) next
    o <- df[df$zaman == a$baslangic, ][match(ortak, df$kisi[df$zaman == a$baslangic]), ]
    s2 <- df[df$zaman == zn, ][match(ortak, df$kisi[df$zaman == zn]), ]
    for (sk_ad in intersect(c("SenMayo", "SenMayo_immun", "SenMayo_immun_disi", "Fridman_SEN_UP"), colnames(df))) {
      dd <- data.frame(y = s2[[sk_ad]] - o[[sk_ad]], adip = s2$Adiposit - o$Adiposit, mak = s2$Makrofaj - o$Makrofaj)
      m0 <- summary(lm(y ~ 1, dd))$coefficients; m1 <- summary(lm(y ~ adip + mak, dd))$coefficients
      bilesim <- rbind(bilesim, data.frame(veri_seti = acc, karsilastirma = paste(zn, "vs", a$baslangic),
        skor = sk_ad, n_cift = nrow(dd), ham_degisim = m0[1, 1], ham_se = m0[1, 2], ham_p = m0[1, 4],
        duzeltilmis_degisim = m1[1, 1], duzeltilmis_se = m1[1, 2], duzeltilmis_p = m1[1, 4],
        adiposit_katsayi_p = m1["adip", 4], makrofaj_katsayi_p = m1["mak", 4]))
    }
  }

  # --- Duyarlılık analizleri: (1) farklı HPA eşikleri, (2) tek gen çıkarma ---
  # Not: ssGSEA'nın aralık normalizasyonu Cohen dz'yi etkilemediği için burada atlanır.
  duy <- NULL
  if (isTRUE(DUYARLILIK)) {
    message("    Duyarlılık analizleri hesaplanıyor...")
    gx <- d$x
    sir <- lapply(seq_len(ncol(gx)), function(j) {
      r <- rank(gx[, j], ties.method = "average"); o <- order(r, decreasing = TRUE)
      list(g = rownames(gx)[o], w = r[o]^0.25) })
    es <- function(gs) vapply(sir, function(sv) {
      hit <- sv$g %in% gs; w <- sv$w * hit
      sum(cumsum(w) / sum(w) - cumsum(!hit) / sum(!hit)) }, 0)
    esl_dz <- function(sc, zn) {
      ortak <- intersect(df$kisi[df$zaman == a$baslangic], df$kisi[df$zaman == zn])
      if (length(ortak) < 3) return(c(NA, NA, NA))
      o <- sc[df$zaman == a$baslangic][match(ortak, df$kisi[df$zaman == a$baslangic])]
      s3 <- sc[df$zaman == zn][match(ortak, df$kisi[df$zaman == zn])]
      f <- s3 - o; c(length(ortak), mean(f) / sd(f), t.test(f)$p.value) }
    satir <- function(analiz, bil, ng, zn, r, rr = NULL) data.frame(
      veri_seti = acc, analiz = analiz, bilesen = bil, n_gen = ng, karsilastirma = paste(zn, "vs", a$baslangic),
      n_cift = r[1], cohen_dz = r[2], p = r[3],
      dz_min = if (is.null(rr)) NA else min(rr[, 2]), dz_max = if (is.null(rr)) NA else max(rr[, 2]),
      p_en_buyuk = if (is.null(rr)) NA else max(rr[, 3]))
    if (!is.null(SABIT_SINIF$immun_payi)) {
      esikler <- list(ana = c(0.6, 0.4), siki = c(0.7, 0.3), gevsek = c(0.5, 0.5))
      ok <- SABIT_SINIF$sinif != "dusuk_ifade"; pay <- SABIT_SINIF$immun_payi; gn <- SABIT_SINIF$gen
      for (nm in names(esikler)) {
        e <- esikler[[nm]]
        kume <- list(immun = intersect(gn[ok & pay >= e[1]], rownames(gx)),
                     stromal = intersect(gn[ok & (if (nm == "gevsek") pay < e[2] else pay <= e[2])], rownames(gx)))
        for (bil in names(kume)) {
          if (length(kume[[bil]]) < 5) next
          sc <- es(kume[[bil]])
          for (zn in dz) { r <- esl_dz(sc, zn); if (!is.na(r[1])) duy <- rbind(duy, satir(paste0("esik_", nm), bil, length(kume[[bil]]), zn, r)) }
        }
      }
    }
    for (bil in c("immun", "stromal")) {
      gs <- if (bil == "immun") setler$SenMayo_immun else setler$SenMayo_immun_disi
      gs <- intersect(gs, rownames(gx)); if (length(gs) < 6) next
      tam <- es(gs); loo <- lapply(gs, function(g) es(setdiff(gs, g)))
      for (zn in dz) {
        r <- esl_dz(tam, zn); if (is.na(r[1])) next
        rr <- t(sapply(loo, esl_dz, zn = zn))
        duy <- rbind(duy, satir("tek_gen_cikarma", bil, length(gs), zn, r, rr))
      }
    }
  }

  # BMI değişimi ile skor değişimi ilişkisi (varsa)
  bmi_kor <- NULL
  if (!is.null(df$BMI) && sum(!is.na(df$BMI)) > 10) {
    for (zn in dz) {
      ortak <- intersect(df$kisi[df$zaman == a$baslangic], df$kisi[df$zaman == zn])
      if (length(ortak) < 8) next
      o <- df[df$zaman == a$baslangic, ][match(ortak, df$kisi[df$zaman == a$baslangic]), ]
      s <- df[df$zaman == zn, ][match(ortak, df$kisi[df$zaman == zn]), ]
      dbmi <- 100 * (s$BMI - o$BMI) / o$BMI
      for (sk_ad in c("SenMayo", "SenMayo_immun", "SenMayo_immun_disi", "Makrofaj")) {
        k <- suppressWarnings(cor.test(dbmi, s[[sk_ad]] - o[[sk_ad]], method = "spearman"))
        bmi_kor <- rbind(bmi_kor, data.frame(veri_seti = acc, karsilastirma = paste(zn, "vs", a$baslangic),
                         baslangic_BMI_ort = mean(o$BMI, na.rm = TRUE), BMI_degisim_yuzde_ort = mean(dbmi, na.rm = TRUE),
                         skor = sk_ad, spearman_rho = unname(k$estimate), p = k$p.value))
      }
    }
  }
  list(sonuc = sonuc, ozgulluk = ozgulluk, kapsam = kapsam, bmi = bmi_kor, kontrol = kontrol_sonuc, bilesim = bilesim, duy = duy)
}


## Gen setleri ve HPA sınıflandırması (ana analizle aynı)
setler <- gen_setlerini_al()
SABIT_SINIF <- tryCatch(hpa_siniflandir(setler$SenMayo), error = function(e) { message("HPA hatası: ", conditionMessage(e)); NULL })
print(table(SABIT_SINIF$sinif))



In [ ]:
# 3) Katılımcı eşleştirmesi
###############################################################################
# D2) Katılımcı eşleştirmesi: kişiye özgü ifade parmak izi
#  1) GSE199063'te (gerçek kişi kodları var) kişi içinde çok kararlı genler seçilir.
#  2) Yöntem GSE199063 içinde doğrulanır (aynı kişi vs farklı kişi).
#  3) GSE59034 içinde "rep" numarasına göre yapılan önce/sonra eşleştirmesi kontrol edilir.
#  4) GSE59034 örnekleri GSE199063 kişileriyle karşılaştırılır (ortak katılımcı var mı?).
###############################################################################
options(width = 220)
dir.create("ek_analiz_cikti", showWarnings = FALSE)
a <- gse_yukle("GSE199063"); b <- gse_yukle("GSE59034")
pa <- a$pd; pb <- b$pd
kisiA <- as.character(pa[["subject_id:ch1"]]); zamA <- as.character(pa[["time_point:ch1"]])
zamB <- sub("obesity status: ", "", as.character(pb[["obesity status:ch1"]]))
if (all(is.na(zamB)) || !length(zamB)) zamB <- ifelse(grepl("before", pb$title), "before", ifelse(grepl("after", pb$title), "after", "never"))
zamB <- ifelse(grepl("before", zamB), "before", ifelse(grepl("after", zamB), "after", "never"))
repB <- sub(".*rep", "", pb$title)
cat("GSE59034 grupları:", paste(names(table(zamB)), table(zamB), collapse = ", "), "\n")

ortak <- intersect(rownames(a$x), rownames(b$x))
xa <- a$x[ortak, ]; xb <- b$x[ortak, ]
tamam <- rowSums(is.na(xa)) == 0 & rowSums(is.na(xb)) == 0
xa <- xa[tamam, ]; xb <- xb[tamam, ]
cat("Ortak gen sayısı:", nrow(xa), "\n")

# 1) GSE199063 kişi içi kararlılık (ICC), yalnızca ameliyatlı kadınlar (>=2 örnek)
icc <- function(x, kisi) {
  tut <- kisi %in% names(which(table(kisi) >= 2)); x <- x[, tut, drop = FALSE]; k <- factor(kisi[tut])
  n0 <- mean(table(k)); m <- t(apply(x, 1, function(v) tapply(v, k, mean)))
  msb <- apply(m, 1, var) * n0; msw <- rowSums((x - m[, as.integer(k)])^2) / (ncol(x) - nlevels(k))
  (msb - msw) / (msb + (n0 - 1) * msw)
}
ia <- icc(xa, kisiA)
sdA <- apply(xa, 1, sd); sdB <- apply(xb, 1, sd)
cat("GSE199063 ICC dağılımı:\n"); print(summary(ia)); print(table(cut(ia, c(-Inf, .5, .7, .8, .9, Inf))))
aday <- names(ia)[!is.na(ia) & ia > 0.8 & sdA > 0.25 & sdB > 0.25]
aday <- aday[order(-ia[aday])][1:min(300, length(aday))]
cat("Parmak izi gen sayısı:", length(aday), "| ilk 30:", paste(head(aday, 30), collapse = ", "), "\n")

za <- t(scale(t(xa[aday, ]))); zb <- t(scale(t(xb[aday, ])))

# 2) Doğrulama GSE199063 içinde
b0 <- which(zamA == "Baseline"); y2 <- which(zamA == "2_year")
o <- intersect(kisiA[b0], kisiA[y2]); i0 <- b0[match(o, kisiA[b0])]; i2 <- y2[match(o, kisiA[y2])]
R <- cor(za[, i0], za[, i2])
ayni <- diag(R); farkli <- R[row(R) != col(R)]
cat(sprintf("\nGSE199063 doğrulama: aynı kişi r medyan %.2f (min %.2f) | farklı kişi r medyan %.2f, maks %.2f | doğru eşleşme (en iyi aday aynı kişi): %d/%d\n",
            median(ayni), min(ayni), median(farkli), max(farkli), sum(apply(R, 1, which.max) == seq_len(nrow(R))), nrow(R)))
esik <- max(farkli)

# 3) GSE59034 içinde önce/sonra eşleştirmesi
ib <- which(zamB == "before"); iaf <- which(zamB == "after"); inv <- which(zamB == "never")
RB <- cor(zb[, ib], zb[, iaf]); dimnames(RB) <- list(paste0("before_rep", repB[ib]), paste0("after_rep", repB[iaf]))
en_iyi <- apply(RB, 1, which.max)
tab3 <- data.frame(once = rownames(RB), en_iyi_sonra = colnames(RB)[en_iyi], r_en_iyi = apply(RB, 1, max),
                   r_ikinci = apply(RB, 1, function(v) sort(v, decreasing = TRUE)[2]),
                   rep_eslesmesi_r = sapply(seq_len(nrow(RB)), function(i) { j <- match(repB[ib][i], repB[iaf]); if (is.na(j)) NA else RB[i, j] }))
tab3$rep_numarasi_ayni_kisi_mi <- sub("before_", "", tab3$once) == sub("after_", "", tab3$en_iyi_sonra)
cat("\nGSE59034: her 'before' örneğinin en çok benzediği 'after' örneği (rep numarası eşleştirmesi doğru mu?)\n")
print(tab3, digits = 3, row.names = FALSE)
cat("rep numarası doğru eşleşme sayısı:", sum(tab3$rep_numarasi_ayni_kisi_mi), "/", nrow(tab3), "\n")
write.csv(tab3, "ek_analiz_cikti/D2_GSE59034_ic_eslesme.csv", row.names = FALSE)
RN <- cor(zb[, inv], zb[, c(ib, iaf)])
cat(sprintf("Kontrollerin ameliyatlı örneklere en yüksek benzerliği: maks r = %.2f\n", max(RN)))

# 4) GSE59034 örnekleri vs GSE199063 kişileri (kişi ortalaması parmak izi)
kA <- split(seq_along(kisiA), kisiA)
profA <- sapply(kA, function(i) rowMeans(za[, i, drop = FALSE]))
zamA_kisi <- sapply(kA, function(i) if (any(zamA[i] == "Control")) "Control" else "RYGB")
C <- cor(zb, profA)                         # satır: GSE59034 örnekleri, sütun: GSE199063 kişileri
sonuc <- data.frame(GSE59034_ornek = paste0(zamB, "_rep", repB), en_iyi_GSE199063 = colnames(C)[apply(C, 1, which.max)],
                    grup_GSE199063 = zamA_kisi[colnames(C)[apply(C, 1, which.max)]],
                    r_en_iyi = apply(C, 1, max), r_ikinci = apply(C, 1, function(v) sort(v, decreasing = TRUE)[2]))
sonuc$fark <- sonuc$r_en_iyi - sonuc$r_ikinci
sonuc$olasi_ayni_kisi <- sonuc$r_en_iyi > esik & sonuc$fark > 0.15
cat(sprintf("\nGSE59034 örnekleri ile GSE199063 kişileri (eşik: r > %.2f ve ikinci adaydan en az 0.15 fazla):\n", esik))
print(sonuc[order(-sonuc$r_en_iyi), ], digits = 3, row.names = FALSE)
cat("\nOlası ortak katılımcı içeren GSE59034 örnek sayısı:", sum(sonuc$olasi_ayni_kisi), "/", nrow(sonuc), "\n")
cat("Eşleşen farklı GSE199063 kişisi sayısı:", length(unique(sonuc$en_iyi_GSE199063[sonuc$olasi_ayni_kisi])), "\n")
write.csv(sonuc, "ek_analiz_cikti/D2_capraz_eslesme.csv", row.names = FALSE)
cat("\nD2 bitti.\n")



In [ ]:
# 4) Normalizasyon ve prob birleştirme duyarlılığı
options(width = 200)
EK <- "ek_analiz_cikti"; dir.create(EK, showWarnings = FALSE)
ekyol <- function(...) file.path(EK, paste0(...))
SKORLAR <- c("SenMayo", "SenMayo_immun", "SenMayo_immun_disi", "Fridman_SEN_UP", "Makrofaj", "Adiposit")
baslik <- function(t) cat("\n\n==================== ", t, " ====================\n", sep = "")
baslik("E) Normalizasyon ve prob birleştirme duyarlılığı")
yedek <- list(CIKTI_KLASORU, DUYARLILIK, RASTGELE_SET_SAYISI, PROB_BIRLESTIRME)
DUYARLILIK <- FALSE; RASTGELE_SET_SAYISI <- 10
E <- list()
calistir <- function(acc, etiket) {
  CIKTI_KLASORU <<- file.path(EK, etiket); dir.create(CIKTI_KLASORU, showWarnings = FALSE)
  r <- tryCatch(suppressMessages(suppressWarnings({ invisible(capture.output(rr <- analiz_et(if (acc %in% RNASEQ_SETLERI) rnaseq_yukle(acc) else gse_yukle(acc), setler))); rr })),
                error = function(e) { message("!!! ", acc, ": ", conditionMessage(e)); NULL })
  if (is.null(r)) return(NULL)
  x <- r$sonuc[r$sonuc$grup == "tum_orneklem" & r$sonuc$skor %in% SKORLAR, c("veri_seti", "karsilastirma", "skor", "n_cift", "cohen_dz", "p_eslestirilmis_t")]
  x$duyarlilik <- etiket; x
}
RNASEQ_TMM <- TRUE;  E[[1]] <- calistir("GSE65540", "TMM"); RNASEQ_TMM <- FALSE
PROB_BIRLESTIRME <- "ortalama"
for (acc in c("GSE199063", "GSE59034", "GSE77962")) E[[length(E) + 1]] <- calistir(acc, "prob_ortalamasi")
CIKTI_KLASORU <- yedek[[1]]; DUYARLILIK <- yedek[[2]]; RASTGELE_SET_SAYISI <- yedek[[3]]; PROB_BIRLESTIRME <- yedek[[4]]
E <- do.call(rbind, E); write.csv(E, ekyol("E_normalizasyon_prob_duyarlilik.csv"), row.names = FALSE); print(E, digits = 3, row.names = FALSE)

## ---------------------------------------------------------------------------

